# Exercise 3: Hosting a Graph Database on AWS

In this exercise, you will learn how to:
1. Use the AWS Boto3 library to launch and connect to an AWS Neptune instance (Amazon's managed graph database)
2. Configure the networking infrastructure (VPC, subnets, security groups) needed for Neptune
3. Explore Gremlin and openCypher query patterns for Neptune
4. Clean up resources to avoid unnecessary charges

## Prerequisites

You will need:

- Your Cloud Resources AWS resources with associated Access Key ID, Secret Access Key, and Session Token
- AWS Boto3 installed and configured
- Basic understanding of graph databases and query languages (from Exercises 1 and 2)

These can be found in the workspace under the tab "Cloud Resources".

## Cost Warning

**IMPORTANT:** This exercise creates AWS resources that may incur charges. Neptune instances start at ~$0.10/hr for the smallest instance. Make sure to run the **Clean Up** section at the end to delete all resources!

## Step 1: Install and Configure boto3

First, let's import os, and boto3, the AWS SDK for Python. Make sure you have AWS credentials configured (via the "Cloud Resources" tab in the workspace).

Copy these somewhere safe, as you will need to paste them into the code segment below. This will be your first task.

In [ ]:
%%capture
%pip install boto3

import boto3
import json
import time
from botocore.exceptions import ClientError

## Configure AWS Credentials

#### Task 1: In the cell below, replace the placeholder values with your actual AWS credentials:

**Required:**
- `aws_access_key_id`: Your AWS Access Key ID
- `aws_secret_access_key`: Your AWS Secret Access Key
- `aws_session_token`: Your AWS Session Token

Note, it's usually not best practices to put passwords and access keys and the like into code. Hardcoding is not secure at all - this is only for demonstration purposes!

In [ ]:
import os
import boto3

# Configure AWS credentials - Replace with your actual credentials
session = boto3.Session(
    aws_access_key_id= '### BEGIN SOLUTION We cannot input this for you! ### END SOLUTION',
    aws_secret_access_key= '### BEGIN SOLUTION We cannot input this for you! ### END SOLUTION',
    aws_session_token= '### BEGIN SOLUTION We cannot input this for you! ### END SOLUTION',  # Optional: only needed for temporary credentials
    region_name='us-east-1'  # Change to your preferred region
)

# Create clients using the configured session
sts = session.client('sts')
boto3.setup_default_session(
    aws_access_key_id=session.get_credentials().access_key,
    aws_secret_access_key=session.get_credentials().secret_key,
    aws_session_token=session.get_credentials().token,
    region_name=session.region_name
)

print("Credentials configured! Run the next cell to verify.")

You should see a statement: 'Credentials configured! Run the next cell to verify.'

You can go ahead and run the next cell.

In [ ]:
# Verify AWS configuration
try:
    identity = sts.get_caller_identity()
    print("✓ AWS Configuration Verified!")
    print(f"Account: {identity['Account']}")
    print(f"User ARN: {identity['Arn']}")
    print(f"User ID: {identity['UserId']}")
except ClientError as e:
    print(f"✗ Error verifying AWS credentials: {e}")
    print("\nPlease check that you've entered valid credentials in the previous cell.")

You should see something like:

✓ AWS Configuration Verified!
Account: (some random account number)

User ARN: arn:aws:sts::(same random account number as above):assumed-role/voclabs/........................

User ID: (random string)

If that's the output you're getting, you're clear to head to the next step.

## Step 2: Network Infrastructure Setup

Just like we did for RDS in Lesson 2, we need to set up networking for our Neptune database. The infrastructure patterns are the same regardless of database type:
- A **VPC** to isolate resources
- **Subnets** across availability zones (Neptune requires at least 2)
- **Security Groups** to control access
- A **DB Subnet Group** to tell Neptune which subnets to use

In [ ]:
# Create EC2 client to query VPC and subnets
ec2 = boto3.client('ec2')

# Get default VPC
vpcs_response = ec2.describe_vpcs(Filters=[{'Name': 'isDefault', 'Values': ['true']}])
if vpcs_response['Vpcs']:
    vpc_id = vpcs_response['Vpcs'][0]['VpcId']
    print(f"Default VPC: {vpc_id}")
else:
    vpcs_response = ec2.describe_vpcs()
    if vpcs_response['Vpcs']:
        vpc_id = vpcs_response['Vpcs'][0]['VpcId']
        print(f"Using VPC: {vpc_id}")
    else:
        raise Exception("No VPC found. Please create a VPC first.")

# Get subnets - Neptune requires subnets in at least 2 AZs
subnets_response = ec2.describe_subnets(Filters=[{'Name': 'vpc-id', 'Values': [vpc_id]}])
subnets = subnets_response['Subnets']

if not subnets:
    raise Exception("No subnets found in VPC. Please create subnets first.")

print(f"\nFound {len(subnets)} subnet(s):")
for subnet in subnets:
    print(f"  - Subnet ID: {subnet['SubnetId']}, AZ: {subnet['AvailabilityZone']}, CIDR: {subnet['CidrBlock']}")

# Neptune requires subnets in at least 2 AZs
az_subnets = {}
for subnet in subnets:
    az = subnet['AvailabilityZone']
    if az not in az_subnets:
        az_subnets[az] = subnet['SubnetId']

subnet_ids = list(az_subnets.values())[:3]
print(f"\nSubnets across {len(subnet_ids)} AZs: {subnet_ids}")

Next, we have to create a security group that allows connections to Neptune. Neptune uses port **8182** for its HTTPS/WebSocket endpoint.

In [ ]:
# Create a security group that allows Neptune connections
security_group_name = 'neptune-exercise-sg'

try:
    sg_response = ec2.create_security_group(
        GroupName=security_group_name,
        Description='Security group for Neptune graph database',
        VpcId=vpc_id
    )
    sg_id = sg_response['GroupId']
    print(f"Security Group created: {sg_id}")

    # Add inbound rule for Neptune (port 8182)
    ec2.authorize_security_group_ingress(
        GroupId=sg_id,
        IpPermissions=[{
            'IpProtocol': 'tcp',
            'FromPort': 8182,
            'ToPort': 8182,
            'IpRanges': [{'CidrIp': '0.0.0.0/0', 'Description': 'Allow Neptune from anywhere'}]
        }]
    )
    print("Inbound rule added: Allow port 8182 from anywhere")

except ClientError as e:
    if e.response['Error']['Code'] == 'InvalidGroup.Duplicate':
        sg_response = ec2.describe_security_groups(
            Filters=[
                {'Name': 'group-name', 'Values': [security_group_name]},
                {'Name': 'vpc-id', 'Values': [vpc_id]}
            ]
        )
        sg_id = sg_response['SecurityGroups'][0]['GroupId']
        print(f"Security Group '{security_group_name}' already exists: {sg_id}")
    else:
        raise e

We need to create a DB Subnet Group for Neptune. This tells Neptune which subnets to use, just like we did for RDS.

In [ ]:
# Create Neptune client
neptune = boto3.client('neptune')

# Create DB Subnet Group for Neptune
db_subnet_group_name = 'neptune-exercise-subnet-group'

try:
    neptune.create_db_subnet_group(
        DBSubnetGroupName=db_subnet_group_name,
        DBSubnetGroupDescription='Subnet group for Neptune graph database exercise',
        SubnetIds=subnet_ids
    )
    print(f"DB Subnet Group '{db_subnet_group_name}' created successfully!")
    print(f"Using subnet(s): {', '.join(subnet_ids)}")
except ClientError as e:
    if 'DBSubnetGroupAlreadyExists' in str(e):
        print(f"DB Subnet Group '{db_subnet_group_name}' already exists. Will use existing group.")
    else:
        raise e

## What is AWS Neptune?

- **Fully managed** graph database service — no server patching, backups, or replication to handle
- **Built-in security** — encryption at rest and in transit, VPC isolation, IAM authentication
- **Continuous backups** — automatic backups to S3 with point-in-time recovery
- **Supports multiple query languages**: Apache TinkerPop Gremlin (property graphs), SPARQL (RDF graphs), and openCypher
- **High availability** — read replicas across AZs, automatic failover
- Best for: Applications needing a managed, highly available graph database integrated with AWS services

## Step 3: Create the Neptune Cluster and Instance

Neptune uses a cluster architecture (similar to Aurora):
- A **Cluster** manages storage and replication
- An **Instance** within the cluster handles compute (reads/writes)

#### Task 2: Choose a name for your Neptune cluster

You can name it whatever you like!

In [ ]:
### BEGIN SOLUTION
neptune_cluster_id = 'graph-exercise-cluster'
### END SOLUTION

# Create Neptune cluster
try:
    response = neptune.create_db_cluster(
        DBClusterIdentifier=neptune_cluster_id,
        Engine='neptune',
        DBSubnetGroupName=db_subnet_group_name,
        VpcSecurityGroupIds=[sg_id],
        StorageEncrypted=True,
        DeletionProtection=False,
        Tags=[{'Key': 'Project', 'Value': 'data-modeling-exercise'}]
    )
    print(f"Neptune cluster creation initiated: {neptune_cluster_id}")
    print(f"  Status: {response['DBCluster']['Status']}")
    print(f"  Engine: {response['DBCluster']['Engine']}")
except ClientError as e:
    if 'DBClusterAlreadyExistsFault' in str(e):
        print(f"Cluster already exists: {neptune_cluster_id}")
    else:
        raise e

### Create a Neptune Instance

Now let's add a compute instance to our cluster. We'll use the smallest available instance type (`db.t3.medium`) to minimize cost.

In [ ]:
neptune_instance_id = 'graph-exercise-instance-1'

try:
    response = neptune.create_db_instance(
        DBInstanceIdentifier=neptune_instance_id,
        DBClusterIdentifier=neptune_cluster_id,
        DBInstanceClass='db.t3.medium',
        Engine='neptune',
        Tags=[{'Key': 'Project', 'Value': 'data-modeling-exercise'}]
    )
    print(f"Neptune instance creation initiated: {neptune_instance_id}")
    print(f"  Instance class: db.t3.medium")
    print(f"  Status: {response['DBInstance']['DBInstanceStatus']}")
except ClientError as e:
    if 'DBInstanceAlreadyExists' in str(e):
        print(f"Instance already exists: {neptune_instance_id}")
    else:
        raise e

### Wait for Neptune to Become Available

Neptune instances typically take **10-15 minutes** to become available. This is similar to RDS — managed database services need time to provision compute, configure networking, and initialize storage.

The following cell defines helper functions and waits for the instance. This is a good time to read ahead about the Gremlin query language!

In [ ]:
def check_neptune_status(instance_id):
    """Check the status of the Neptune instance"""
    try:
        response = neptune.describe_db_instances(DBInstanceIdentifier=instance_id)
        status = response['DBInstances'][0]['DBInstanceStatus']
        return status
    except ClientError as e:
        print(f"Error checking Neptune status: {e}")
        return None

def wait_for_neptune_available(instance_id, max_wait_time=1200, check_interval=30):
    """Wait for the Neptune instance to become available"""
    elapsed_time = 0
    print("Waiting for Neptune instance to become available...")
    print("   (This typically takes 10-15 minutes)")

    while elapsed_time < max_wait_time:
        status = check_neptune_status(instance_id)
        print(f"Status after {elapsed_time}s: {status}")

        if status == 'available':
            print("Neptune instance is now available!")
            response = neptune.describe_db_instances(DBInstanceIdentifier=instance_id)
            endpoint = response['DBInstances'][0]['Endpoint']['Address']
            port = response['DBInstances'][0]['Endpoint']['Port']
            return endpoint, port

        time.sleep(check_interval)
        elapsed_time += check_interval

    print("Timeout waiting for Neptune instance.")
    return None, None

def get_neptune_endpoint(instance_id):
    """Get the endpoint address of the Neptune instance"""
    try:
        response = neptune.describe_db_instances(DBInstanceIdentifier=instance_id)
        endpoint = response['DBInstances'][0]['Endpoint']['Address']
        port = response['DBInstances'][0]['Endpoint']['Port']
        return endpoint, port
    except (ClientError, KeyError) as e:
        print(f"Error retrieving endpoint: {e}")
        return None, None

print("Helper functions defined successfully!")

# Wait for the instance to become available
neptune_endpoint, neptune_port = wait_for_neptune_available(neptune_instance_id)
if neptune_endpoint:
    print(f"\nNeptune Endpoint: {neptune_endpoint}")
    print(f"Port: {neptune_port}")

## Step 4: Query Neptune

Neptune supports multiple query languages. In this step, we'll explore two:

1. **Gremlin** (Apache TinkerPop) — a traversal-based graph query language
2. **openCypher** — the open-source version of the Cypher query language you used in Exercises 1 and 2

#### Task 3: Review the Gremlin query examples below

Gremlin uses a traversal-based syntax:
- `g.addV('label')` — Add a vertex (node)
- `g.V()` — Get all vertices
- `g.addE('label')` — Add an edge (relationship)
- `.property('key', 'value')` — Set properties
- `.has('key', 'value')` — Filter by property

Note: Neptune is VPC-only by default. To query from outside the VPC, you would need a bastion host, VPN, or Neptune Notebook. For this exercise, we demonstrate the query patterns conceptually.

In [ ]:
import urllib.request
import urllib.parse

def gremlin_query(endpoint, port, query):
    """Send a Gremlin query to Neptune via HTTP."""
    url = f"https://{endpoint}:{port}/gremlin"
    data = json.dumps({'gremlin': query}).encode('utf-8')
    req = urllib.request.Request(
        url,
        data=data,
        headers={'Content-Type': 'application/json'}
    )
    try:
        with urllib.request.urlopen(req) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"Query error: {e}")
        return None

# Example Gremlin queries (these would run if connected within the VPC)
print("Neptune Gremlin Query Examples:")
print("="*50)

gremlin_examples = [
    {
        'description': 'Add a Customer vertex',
        'query': "g.addV('Customer').property('name', 'Alice').property('status', 'gold')"
    },
    {
        'description': 'Add a Product vertex',
        'query': "g.addV('Product').property('name', 'Small Gear').property('price', 49.99)"
    },
    {
        'description': 'Create a PURCHASED edge',
        'query': "g.V().has('Customer','name','Alice').addE('PURCHASED').to(g.V().has('Product','name','Small Gear')).property('quantity', 5)"
    },
    {
        'description': 'Find all customers who purchased a product',
        'query': "g.V().hasLabel('Product').has('name','Small Gear').in('PURCHASED').values('name')"
    },
    {
        'description': 'Count all vertices by label',
        'query': "g.V().groupCount().by(label)"
    }
]

for ex in gremlin_examples:
    print(f"\n{ex['description']}:")
    print(f"   {ex['query']}")

### Neptune openCypher Support

Neptune also supports **openCypher** queries — the same Cypher syntax you used in Exercises 1 and 2! This means your existing Cypher knowledge transfers directly.

#### Task 4: Review the openCypher query examples below

The same MATCH patterns work on Neptune's `/openCypher` endpoint.

In [ ]:
# openCypher queries work on Neptune's /openCypher endpoint
# These are the same Cypher queries from Exercises 1 and 2!

### BEGIN SOLUTION
opencypher_examples = [
    {
        'description': 'Create a Customer node',
        'query': "CREATE (c:Customer {name: 'Alice', status: 'gold', email: 'alice@company.com'})"
    },
    {
        'description': 'Find all gold customers',
        'query': "MATCH (c:Customer {status: 'gold'}) RETURN c.name, c.email"
    },
    {
        'description': 'Create a purchase relationship',
        'query': "MATCH (c:Customer {name: 'Alice'}), (p:Product {name: 'Small Gear'}) CREATE (c)-[:PURCHASED {qty: 5}]->(p)"
    },
    {
        'description': 'Find products purchased by a customer',
        'query': "MATCH (c:Customer {name: 'Alice'})-[:PURCHASED]->(p:Product) RETURN p.name, p.price"
    }
]
### END SOLUTION

print("Neptune openCypher Query Examples:")
print("="*50)
for ex in opencypher_examples:
    print(f"\n{ex['description']}:")
    print(f"   {ex['query']}")

print("\nNotice: These are identical to the Cypher queries from Exercises 1 and 2!")
print("Your Cypher skills transfer directly to Neptune.")

### Check Neptune Cluster Status

Let's verify our Neptune cluster details and explore its configuration.

In [ ]:
# Check Neptune cluster status
try:
    cluster_info = neptune.describe_db_clusters(
        DBClusterIdentifier=neptune_cluster_id
    )['DBClusters'][0]

    print("Neptune Cluster Details:")
    print("=" * 50)
    print(f"  Cluster ID:    {cluster_info['DBClusterIdentifier']}")
    print(f"  Status:        {cluster_info['Status']}")
    print(f"  Engine:        {cluster_info['Engine']}")
    print(f"  Engine Ver:    {cluster_info['EngineVersion']}")
    print(f"  Endpoint:      {cluster_info['Endpoint']}")
    print(f"  Port:          {cluster_info['Port']}")
    print(f"  Encrypted:     {cluster_info['StorageEncrypted']}")
    print(f"  Backup Ret.:   {cluster_info['BackupRetentionPeriod']} days")

    # Instance details
    instance_info = neptune.describe_db_instances(
        DBInstanceIdentifier=neptune_instance_id
    )['DBInstances'][0]

    print(f"\nNeptune Instance Details:")
    print(f"  Instance ID:   {instance_info['DBInstanceIdentifier']}")
    print(f"  Instance Class:{instance_info['DBInstanceClass']}")
    print(f"  Status:        {instance_info['DBInstanceStatus']}")
    print(f"  AZ:            {instance_info['AvailabilityZone']}")

except ClientError as e:
    print(f"Could not describe cluster: {e}")

## Step 5: Clean Up Resources

**IMPORTANT:** To avoid unnecessary AWS charges, make sure to clean up the resources you created in this exercise.

Resources will be deleted in reverse order:
1. Neptune instance
2. Neptune cluster
3. DB subnet group
4. Security group

In [ ]:
# Delete Neptune instance
try:
    neptune.delete_db_instance(
        DBInstanceIdentifier=neptune_instance_id,
        SkipFinalSnapshot=True
    )
    print(f"Neptune instance deletion initiated: {neptune_instance_id}")
except ClientError as e:
    if 'DBInstanceNotFound' in str(e):
        print(f"Instance already deleted: {neptune_instance_id}")
    else:
        print(f"Error: {e}")

In [ ]:
# Wait for instance to be deleted before deleting the cluster
def wait_for_neptune_deleted(instance_id, max_wait_time=900, check_interval=30):
    """Wait for Neptune instance to be completely deleted"""
    elapsed_time = 0
    print("Waiting for Neptune instance to be deleted...")

    while elapsed_time < max_wait_time:
        try:
            response = neptune.describe_db_instances(
                DBInstanceIdentifier=instance_id
            )
            status = response['DBInstances'][0]['DBInstanceStatus']
            print(f"Status after {elapsed_time}s: {status}")
        except ClientError as e:
            if 'DBInstanceNotFound' in str(e):
                print(f"Status after {elapsed_time}s: Instance not found - deletion complete!")
                return True
            else:
                print(f"Error checking status: {e}")
                return False

        time.sleep(check_interval)
        elapsed_time += check_interval

    print("Timeout waiting for Neptune instance deletion.")
    return False

# Wait for deletion to complete
wait_for_neptune_deleted(neptune_instance_id)

In [ ]:
# Delete Neptune cluster
try:
    neptune.delete_db_cluster(
        DBClusterIdentifier=neptune_cluster_id,
        SkipFinalSnapshot=True
    )
    print(f"Neptune cluster deletion initiated: {neptune_cluster_id}")
except ClientError as e:
    if 'DBClusterNotFoundFault' in str(e):
        print(f"Cluster already deleted: {neptune_cluster_id}")
    else:
        print(f"Error: {e}")

print("\nWaiting 60 seconds for cluster deletion to propagate...")
time.sleep(60)
print("Done waiting.")

Once the Neptune cluster is deleted, we can clean up the remaining AWS resources (Subnet Group and Security Group).

In [ ]:
# Delete DB subnet group
try:
    neptune.delete_db_subnet_group(
        DBSubnetGroupName=db_subnet_group_name
    )
    print(f"DB Subnet Group '{db_subnet_group_name}' deleted successfully!")
except ClientError as e:
    if 'DBSubnetGroupNotFoundFault' in str(e):
        print(f"Subnet group already deleted")
    else:
        print(f"Error: {e}")

# Delete security group
try:
    ec2.delete_security_group(GroupId=sg_id)
    print(f"Security Group '{security_group_name}' deleted successfully!")
except ClientError as e:
    if 'InvalidGroup.NotFound' in str(e):
        print(f"Security group already deleted")
    elif 'DependencyViolation' in str(e):
        print(f"Security group still in use — Neptune may still be deleting.")
        print(f"Wait a few minutes and re-run this cell.")
    else:
        print(f"Error: {e}")

Congratulations! You now know how to launch and manage an AWS Neptune graph database instance using Amazon's Boto3 Python library. The commands with the CLI are very similar.

## Summary

### What You Learned

1. **AWS Neptune** — Amazon's fully managed graph database with built-in security, continuous backups, and support for Gremlin, SPARQL, and openCypher query languages
2. **Infrastructure Patterns** — VPCs, security groups, and subnet groups are the same regardless of database type (RDBMS, NoSQL, or Graph)
3. **Query Language Portability** — Neptune's openCypher support means your Cypher skills from Exercises 1 & 2 transfer directly
4. **Gremlin** — An alternative traversal-based query language for property graphs

### Key Takeaways

- Graph databases on AWS follow the **same operational patterns** as any other managed database (just like RDS in Lesson 2)
- **Neptune** is ideal when you want zero-ops management and native AWS integration
- The hosting challenges (networking, security, backups, scaling) are **universal** — they don't change based on data model
- Neptune supports **multiple query languages** (Gremlin, SPARQL, openCypher), giving you flexibility

## Bonus Reference: AWS CLI Equivalents

Everything we did with boto3 can also be done via the AWS CLI:

```bash
# Create Neptune cluster
aws neptune create-db-cluster \
    --db-cluster-identifier graph-exercise-cluster \
    --engine neptune \
    --db-subnet-group-name neptune-exercise-subnet-group \
    --vpc-security-group-ids sg-xxxxx

# Create Neptune instance
aws neptune create-db-instance \
    --db-instance-identifier graph-exercise-instance-1 \
    --db-cluster-identifier graph-exercise-cluster \
    --db-instance-class db.t3.medium \
    --engine neptune

# Check cluster status
aws neptune describe-db-clusters \
    --db-cluster-identifier graph-exercise-cluster

# Delete instance
aws neptune delete-db-instance \
    --db-instance-identifier graph-exercise-instance-1 \
    --skip-final-snapshot

# Delete cluster
aws neptune delete-db-cluster \
    --db-cluster-identifier graph-exercise-cluster \
    --skip-final-snapshot
```